# Ingestion Pipeline Explorer

Step-by-step cells to inspect the ingestion pipeline and DoclingHybridChunker output.

**Sections**
1. Setup & config
2. DoclingHybridChunker — fallback path (plain text / markdown)
3. DoclingHybridChunker — primary path (PDF/DOCX via DocumentConverter → contextualized chunks)
4. Embedding a batch of chunks
5. Full single-file ingestion (no DB write)
6. Full pipeline — ingest all documents into PostgreSQL

## 1. Setup

In [20]:
import sys, os
import markdown
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath('..'))

import asyncio, textwrap, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../.env')

True

In [ ]:
try:
    from rag.config.settings import load_settings
    settings = load_settings()
    print(f"LLM model       : {settings.llm_model}")
    print(f"Embedding model : {settings.embedding_model}  ({settings.embedding_dimension}-dim)")
    print(f"DB              : {settings.database_url[:40]}...")
except Exception as e:
    print(f"[ERROR] Settings failed to load: {e}")
    print("Check your .env file and that rag/ is on sys.path.")

## 2. DoclingHybridChunker — Fallback Path (plain text)

When there is no `DoclingDocument` (e.g. a raw `.txt` string), the chunker falls back to sliding-window with sentence-boundary detection.  
Useful for understanding the baseline before the structured path.

In [ ]:
try:
    from rag.ingestion.models import ChunkingConfig
    from rag.ingestion.chunkers.docling import create_chunker

    config = ChunkingConfig(
        chunk_size=500,
        chunk_overlap=100,
        max_tokens=512,
    )
    chunker = create_chunker(config)
    print("Chunker ready")
except Exception as e:
    print(f"[ERROR] Chunker init failed: {e}")

In [ ]:
try:
    doc_path = Path('../rag/documents/team-handbook.md')
    content = doc_path.read_text(encoding='utf-8')
    print(f"Document: {doc_path.name}")
    print(f"Length  : {len(content):,} chars")
    print("---")
    print(content[:600])
except Exception as e:
    print(f"[ERROR] Could not read document: {e}")

In [ ]:
try:
    # Chunk without DoclingDocument → fallback path
    chunks_fallback = await chunker.chunk_document(
        content=content,
        title="Team Handbook",
        source=str(doc_path),
        # no docling_doc= → triggers fallback
    )
    print(f"Chunks produced : {len(chunks_fallback)}")
    print(f"Chunk method    : {chunks_fallback[0].metadata.get('chunk_method')}")
except Exception as e:
    print(f"[ERROR] Chunking failed: {e}")

In [ ]:
try:
    # Inspect fallback chunks
    for i, c in enumerate(chunks_fallback):
        print(f"── Chunk {i}  tokens={c.token_count}  chars={c.start_char}–{c.end_char}")
        print(textwrap.indent(c.content[:300], '   '))
        print()
except Exception as e:
    print(f"[ERROR] {e}")

## 3. DoclingHybridChunker — Primary Path (PDF → DoclingDocument → contextualized chunks)

When Docling's `DocumentConverter` processes a PDF or DOCX, it produces a `DoclingDocument` — a structured
representation with section headings, tables, etc.  
The `HybridChunker` uses that structure to split at natural boundaries, then `contextualize()` prepends
the heading hierarchy to every chunk so each chunk is self-contained for embedding.

In [ ]:
from docling.document_converter import DocumentConverter

pdf_path = Path('../rag/documents/technical-architecture-guide.pdf')
print(f"Converting: {pdf_path.name} ...")

try:
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    docling_doc = result.document
    markdown_text = result.document.export_to_markdown()
    _conversion_ok = True
    print(f"Conversion done")
    print(f"Markdown length : {len(markdown_text):,} chars")
    print("---")
    print(markdown_text[:800])
except Exception as e:
    _conversion_ok = False
    docling_doc = None
    markdown_text = None
    print(f"[ERROR] Conversion failed: {e}")
    print("Downstream cells (chunking, side-by-side, embedding) will be skipped.")

In [ ]:
if not _conversion_ok:
    print("Skipping: PDF conversion did not succeed. Re-run the cell above first.")
else:
    # Chunk with DoclingDocument → primary HybridChunker path
    chunks_hybrid = await chunker.chunk_document(
        content=markdown_text,
        title="Technical Architecture Guide",
        source=str(pdf_path),
        docling_doc=docling_doc,   # ← the structured doc
    )

    print(f"Chunks produced : {len(chunks_hybrid)}")
    print(f"Chunk method    : {chunks_hybrid[0].metadata.get('chunk_method')}")
    print(f"Has context     : {chunks_hybrid[0].metadata.get('has_context')}")

In [ ]:
if not _conversion_ok:
    print("Skipping: PDF conversion did not succeed.")
else:
    # Show all chunks — heading hierarchy + body clearly visible
    for i, c in enumerate(chunks_hybrid):
        print(f"{'='*60}")
        print(f" Chunk {i:>2}  |  tokens={c.token_count}  |  {c.metadata.get('chunk_method')}")
        print(f"{'='*60}")
        print(c.content)
        print()

In [ ]:
if not _conversion_ok:
    print("Skipping: PDF conversion did not succeed.")
else:
    import statistics
    tokens = [c.token_count for c in chunks_hybrid if c.token_count]
    print(f"Chunks : {len(tokens)}")
    print(f"Min    : {min(tokens)}")
    print(f"Max    : {max(tokens)}")
    print(f"Mean   : {statistics.mean(tokens):.1f}")
    print(f"Median : {statistics.median(tokens):.1f}")
    print(f"Stdev  : {statistics.stdev(tokens):.1f}")

### 3a. Side-by-side: with vs without contextualization

To see exactly what `contextualize()` adds, we chunk the same document but call `chunker.chunk()` directly (no contextualization) and compare.

In [ ]:
if not _conversion_ok:
    print("Skipping: PDF conversion did not succeed.")
else:
    from docling.chunking import HybridChunker as DoclingHybridChunker

    raw_chunks = list(chunker.chunker.chunk(dl_doc=docling_doc))

    print(f"Raw chunks (no context): {len(raw_chunks)}")
    print()

    # Pick a chunk mid-document where heading context matters
    idx = min(3, len(raw_chunks) - 1)

    raw_text = raw_chunks[idx].text if hasattr(raw_chunks[idx], 'text') else str(raw_chunks[idx])
    ctx_text = chunker.chunker.contextualize(chunk=raw_chunks[idx])

    print(f"Chunk index: {idx}")
    print()
    print("WITHOUT contextualize():")
    print("-" * 40)
    print(raw_text[:500])
    print()
    print("WITH contextualize():")
    print("-" * 40)
    print(ctx_text[:500])

## 4. Embedding a Batch of Chunks

In [ ]:
try:
    from rag.ingestion.embedder import EmbeddingGenerator
    embedder = EmbeddingGenerator()
    print(f"Embedding model : {embedder.model}")
    print(f"Dimension       : {embedder.get_embedding_dimension()}")
except Exception as e:
    print(f"[ERROR] EmbeddingGenerator init failed: {e}")

In [ ]:
if not _conversion_ok:
    print("Skipping: PDF conversion did not succeed. Using fallback chunks instead.")
    sample_chunks = chunks_fallback[:5]
else:
    sample_chunks = chunks_hybrid[:5]

print(f"Embedding {len(sample_chunks)} chunks...")
embedded_chunks = await embedder.embed_chunks(sample_chunks)

for c in embedded_chunks:
    emb = c.embedding
    print(f"  Chunk {c.index:>2}  tokens={c.token_count:>3}  "
          f"embedding dim={len(emb)}  "
          f"first5={[round(v,4) for v in emb[:5]]}")

## 5. Full Single-File Ingestion (no DB write)

Run the complete pipeline on one file — convert → title extract → chunk → embed — and inspect the final `IngestionResult` without touching the database.

In [ ]:
try:
    from rag.ingestion.pipeline import DocumentIngestionPipeline, IngestionConfig

    pipe_config = IngestionConfig(
        chunk_size=500,
        chunk_overlap=100,
        max_tokens=512,
    )

    # Create pipeline but don't initialize DB yet
    pipeline = DocumentIngestionPipeline(
        config=pipe_config,
        documents_folder='../rag/documents',
        clean_before_ingest=False,
    )
    print("Pipeline created (DB not yet initialized)")
except Exception as e:
    print(f"[ERROR] Pipeline creation failed: {e}")

In [ ]:
try:
    # Run just the document-processing steps (no store.save / store.add)
    # We call the private methods directly so we can inspect intermediate output

    target = Path('../rag/documents/mission-and-goals.md')

    # Step 1: read
    raw_content, docling_doc_result = await pipeline._read_document(target)
    print(f"Raw content length : {len(raw_content):,} chars")
    print(f"DoclingDocument    : {'yes' if docling_doc_result else 'no (fallback)'}")
    print()
    print(raw_content[:400])
except Exception as e:
    print(f"[ERROR] _read_document failed: {e}")

In [ ]:
try:
    # Step 2: extract title
    title = pipeline._extract_title(raw_content, target)
    print(f"Title: {title}")
except Exception as e:
    print(f"[ERROR] _extract_title failed: {e}")

In [ ]:
try:
    # Step 3: metadata (includes file hash)
    metadata = pipeline._extract_document_metadata(target)
    print(json.dumps(metadata, indent=2, default=str))
except Exception as e:
    print(f"[ERROR] _extract_document_metadata failed: {e}")

In [ ]:
try:
    # Step 4: chunk
    chunks = await pipeline.chunker.chunk_document(
        content=raw_content,
        title=title,
        source=str(target),
        metadata=metadata,
        docling_doc=docling_doc_result,
    )
    print(f"Chunks: {len(chunks)}")
    for c in chunks:
        print(f"  [{c.index}] tokens={c.token_count}  method={c.metadata.get('chunk_method')}")
        print(f"      {c.content[:120].strip()!r}")
except Exception as e:
    print(f"[ERROR] chunk_document failed: {e}")

In [ ]:
try:
    # Step 5: embed
    embedded = await pipeline.embedder.embed_chunks(chunks)
    print(f"Embedded {len(embedded)} chunks")
    for c in embedded:
        print(f"  [{c.index}] dim={len(c.embedding)}  norm≈{sum(v**2 for v in c.embedding)**0.5:.3f}")
except Exception as e:
    print(f"[ERROR] embed_chunks failed: {e}")

## 6. Full Pipeline — Ingest All Documents into PostgreSQL

This writes to the database. Run with `clean_before_ingest=True` to wipe and re-ingest, or `False` for incremental.

In [ ]:
try:
    from rag.ingestion.pipeline import DocumentIngestionPipeline, IngestionConfig

    full_config = IngestionConfig(
        chunk_size=1000,
        chunk_overlap=200,
        max_tokens=512,
    )

    full_pipeline = DocumentIngestionPipeline(
        config=full_config,
        documents_folder='../rag/documents',
        clean_before_ingest=True,   # ← set False for incremental
    )
    print("Ready. Run the next cell to ingest.")
except Exception as e:
    print(f"[ERROR] Full pipeline creation failed: {e}")

In [ ]:
try:
    # Progress callback so we can see what's happening
    def progress(current, total, filename, status):
        bar = '█' * current + '░' * (total - current)
        print(f"[{bar}] {current}/{total}  {status:10}  {Path(filename).name}")

    results = await full_pipeline.ingest_documents(progress_callback=progress)
    await full_pipeline.close()
except Exception as e:
    print(f"[ERROR] Ingestion failed: {e}")
    raise

In [ ]:
try:
    total_chunks = sum(r.chunks_created for r in results)
    print(f"Documents ingested : {len(results)}")
    print(f"Total chunks       : {total_chunks}")
    print()
    print(f"{'Document':<45} {'Chunks':>6} {'Time (ms)':>10}")
    print('-' * 65)
    for r in results:
        print(f"{r.title:<45} {r.chunks_created:>6} {r.processing_time_ms:>10.0f}")
except Exception as e:
    print(f"[ERROR] Summary failed: {e}")